# Sailing Training

This notebook trains two neural networks on sailing video frames and compares them.

**Prerequisite:** run `Sailing Labeling.ipynb` first to produce `train.csv`, `val.csv`, `test.csv` and `class_info.json`.

| Step | What happens |
|---|---|
| 0. Setup | Install packages, imports, mount Drive, constants |
| 1. Dataset | Custom Keras Sequence + Albumentations augmentation |
| 2. Models | Model A: single-head (predicts boat, side, direction together) vs Model B: triple-head (cascaded) |
| 3. Training | Focal loss, class weights, EarlyStopping, backbone warm-up |
| 4. Evaluation | Confusion matrices, metric comparison between the two models |


## 0. Setup

In [ ]:
# Install required packages if not already present
import subprocess, sys, importlib

# Dictionary: Python module name -> pip package name
packages = {
    'tensorflow':    'tensorflow',
    'keras':         'keras',
    'cv2':           'opencv-python-headless',
    'albumentations':'albumentations',
    'tqdm':          'tqdm',
    'matplotlib':    'matplotlib',
    'sklearn':       'scikit-learn',
    'pandas':        'pandas',
}

for module_name, pip_name in packages.items():
    try:
        importlib.import_module(module_name)       # try to import; if it works, skip
        print(f'{pip_name} already installed')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', pip_name, '--quiet'],
                       capture_output=True)         # install silently
        print(f'{pip_name} installed')

print('All dependencies ready')


In [ ]:
import os
import json
import csv
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2                                             # OpenCV: image reading and resizing
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import keras
from keras import layers, regularizers, callbacks      # Keras building blocks
from keras.applications import EfficientNetV2B2        # pretrained backbone
import tensorflow as tf
import albumentations as A                             # augmentation library

print(f'Keras : {keras.__version__}  |  TF : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs  : {[g.name for g in gpus] if gpus else "CPU only"}')


In [ ]:
# Mount Google Drive so we can read the data and save checkpoints
from google.colab import drive
drive.mount('/content/drive')

# --- Folder paths -----------------------------------------------------------
DRIVE_ROOT     = Path('/content/drive/MyDrive')
PROJECT_ROOT   = DRIVE_ROOT / 'Sailing2'
VIDEO_ROOT     = PROJECT_ROOT / 'video_temp3'
DATA_DIR       = PROJECT_ROOT / 'data_temp3'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints_temp3'
RESULTS_DIR    = PROJECT_ROOT / 'results_temp3'

# Create output folders if they do not exist yet
for folder in [CHECKPOINT_DIR, RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Drive mounted')
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
# Load class names, index mappings, and image size from the JSON file
# produced by Sailing Labeling.ipynb
with open(DATA_DIR / 'class_info.json') as f:
    class_info = json.load(f)

# Class label lists (order matters: index 0 = first class, etc.)
DIRECTION_CLASSES = class_info['direction_classes']
SIDE_CLASSES      = class_info['side_classes']
BOAT_TYPE_CLASSES = class_info['boat_type_classes']

# Dictionaries to convert class name -> integer index
DIR2IDX  = class_info['dir2idx']
SIDE2IDX = class_info['side2idx']
BOAT2IDX = class_info['boat2idx']

# Reverse dictionaries: integer index -> class name
IDX2DIR  = {int(v): k for k, v in DIR2IDX.items()}
IDX2SIDE = {int(v): k for k, v in SIDE2IDX.items()}
IDX2BOAT = {i: c for i, c in enumerate(BOAT_TYPE_CLASSES)}

# Image dimensions (square, e.g. 224x224)
IMG_SIZE = class_info.get('img_size', 224)

# --- Training hyperparameters -----------------------------------------------
BATCH_SIZE      = 32        # number of images per gradient update
NUM_EPOCHS      = 60        # maximum training epochs before stopping
LR_INITIAL      = 1e-3      # learning rate while backbone is frozen
LR_FINE         = 1e-4      # learning rate after unfreezing backbone
WEIGHT_DECAY    = 1e-4      # L2 regularization strength
DROPOUT         = 0.3       # fraction of activations to drop during training
LABEL_SMOOTHING = 0.1       # softens one-hot targets to improve generalization
FOCAL_GAMMA     = 2.0       # focal loss focusing parameter (higher = more focus on hard examples)
EARLY_STOP_PAT  = 8         # stop training if val loss does not improve for this many epochs
FREEZE_EPOCHS   = 5         # how many epochs to train with the backbone frozen

# --- Loss weights per head (must sum to 1.0) ---------------------------------
# Direction is harder to predict so it gets the highest weight
BOAT_LOSS_W = 0.2
SIDE_LOSS_W = 0.3
DIR_LOSS_W  = 0.5

# --- ImageNet normalization statistics --------------------------------------
# EfficientNet was pretrained on ImageNet, so we normalize using ImageNet stats
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

# Fix the random seed so results are reproducible
keras.utils.set_random_seed(42)

print('Constants loaded')
print(f'Direction classes ({len(DIRECTION_CLASSES)}): {DIRECTION_CLASSES}')
print(f'Side classes      ({len(SIDE_CLASSES)}): {SIDE_CLASSES}')
print(f'Boat type classes ({len(BOAT_TYPE_CLASSES)}): {BOAT_TYPE_CLASSES}')


## 1. Dataset, Augmentation and Data Generators

We use **Albumentations** for augmentation rather than Keras preprocessing layers because:
- Albumentations provides weather simulation transforms (fog, rain, sun flare) that have no Keras equivalent
- Albumentations provides MotionBlur and GaussNoise which are absent from Keras
- Keras augmentation layers are useful when you want GPU-accelerated augmentation baked into the model graph, but since we augment on CPU inside the generator the advantage disappears

The `SailingSequence` class extends `keras.utils.Sequence`, which is the standard Keras interface for writing a custom data loader. Keras calls `__getitem__(batch_index)` to get each batch and `on_epoch_end()` to reshuffle between epochs.


In [ ]:
# --- Albumentations augmentation pipelines ----------------------------------
# build_train_transform() returns a pipeline used only during training.
# build_val_transform() returns only a resize, used during validation and test.

def build_train_transform():
    # A.Compose chains multiple transforms; they are applied in order
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),                            # ensure correct spatial size
        A.Rotate(limit=30, border_mode=cv2.BORDER_REFLECT_101, p=0.6),  # random rotation up to 30 degrees
        A.Perspective(scale=(0.04, 0.10), p=0.4),                # simulate camera angle change
        A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE),
                            scale=(0.75, 1.0), ratio=(0.85, 1.18), p=0.5),  # random crop and scale
        A.Affine(translate_percent=0.05, scale=(0.9, 1.1), p=0.3),  # small translations and scalings
        A.ColorJitter(brightness=0.4, contrast=0.4,
                      saturation=0.3, hue=0.1, p=0.7),           # random color distortion
        A.HueSaturationValue(15, 30, 20, p=0.5),                 # shift hue, saturation, value channels
        A.RandomBrightnessContrast(0.3, 0.3, p=0.5),             # random brightness and contrast
        A.MotionBlur(blur_limit=(3, 9), p=0.3),                  # simulate camera motion blur
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),             # add random Gaussian noise
        A.ImageCompression(quality_lower=60, quality_upper=95, p=0.2),  # simulate JPEG artifacts
        A.RandomSunFlare(flare_roi=(0, 0, 1, 0.4),
                         num_flare_circles_lower=3,
                         num_flare_circles_upper=6,
                         src_radius=80,
                         src_color=(255, 255, 200), p=0.15),     # sun glare on water/sky
        A.RandomFog(fog_coef_lower=0.05,
                    fog_coef_upper=0.25, alpha_coef=0.1, p=0.15),  # foggy weather
        A.RandomRain(slant_lower=-10, slant_upper=10,
                     drop_length=10, drop_width=1,
                     drop_color=(200, 200, 200),
                     blur_value=3, brightness_coefficient=0.85,
                     rain_type='drizzle', p=0.10),               # rain droplets
        A.RandomToneCurve(scale=0.15, p=0.3),                    # small tone curve adjustment
        A.ToGray(p=0.02),                                         # rarely convert to grayscale
        A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                        min_holes=1, min_height=8, min_width=8,
                        fill_value=0, p=0.25),                   # randomly black-out rectangles
    ])


def build_val_transform():
    # Validation and test images are only resized, never augmented
    return A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])


def preprocess_image(img_rgb, augment_fn=None):
    """
    Apply augmentation (if provided) and then normalize with ImageNet stats.
    Input:  uint8 RGB image (H x W x 3)
    Output: float32 image normalized for EfficientNet
    """
    if augment_fn is not None:
        img_rgb = augment_fn(image=img_rgb)['image']   # albumentations returns a dict
    img = img_rgb.astype(np.float32) / 255.0           # scale pixels to [0, 1]
    img = (img - IMAGENET_MEAN) / IMAGENET_STD         # normalize per-channel
    return img.astype(np.float32)


print('Augmentation pipelines ready')


In [ ]:
# --- Class weights -----------------------------------------------------------
# Classes with fewer training samples get higher weights so the model
# does not ignore rare classes.
# class_info stores weights by class name; here we reindex by integer index.
boat_type_weights = {i: class_info['boat_type_weights'][c]
                     for i, c in enumerate(BOAT_TYPE_CLASSES)}
dir_weights       = {i: class_info['dir_weights'][c]
                     for i, c in enumerate(DIRECTION_CLASSES)}
side_weights      = {i: class_info['side_weights'][c]
                     for i, c in enumerate(SIDE_CLASSES)}


class SailingSequence(keras.utils.Sequence):
    """
    Custom Keras data loader for sailing frames.
    Inherits from keras.utils.Sequence so Keras knows how to call it during training.

    Each call to __getitem__ returns one batch of (images, labels, sample_weights).
    Labels are returned as a dictionary matching the output layer names of the model.
    """

    def __init__(self, manifest_path, batch_size=BATCH_SIZE, augment=False, shuffle=True):
        super().__init__()
        self.df         = pd.read_csv(manifest_path)   # load the frame manifest CSV
        self.batch_size = batch_size
        self.shuffle    = shuffle
        # Use the full augmentation pipeline during training, just resize for val/test
        self.aug_fn     = build_train_transform() if augment else build_val_transform()
        self.indices    = np.arange(len(self.df))      # array of row indices into self.df
        if shuffle:
            np.random.shuffle(self.indices)             # shuffle at the start

    def __len__(self):
        # Total number of batches per epoch (round up so no sample is skipped)
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, batch_idx):
        # Select the row indices for this batch
        idxs = self.indices[batch_idx * self.batch_size:(batch_idx + 1) * self.batch_size]

        images, boat_labels, side_labels, dir_labels, sample_weights = [], [], [], [], []

        for i in idxs:
            row = self.df.iloc[i]

            # Read the image from disk
            img = cv2.imread(row['abs_path'])
            if img is None:
                # If the file is missing, substitute a black image so training does not crash
                img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
            else:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)   # OpenCV reads BGR; convert to RGB

            images.append(preprocess_image(img, self.aug_fn))

            # Integer labels for each classification head
            boat_labels.append(int(row['boat_type_idx']))
            side_labels.append(int(row['side_idx']))
            dir_labels.append(int(row['dir_idx']))

            # Combine per-head class weights into a single sample weight
            w = (BOAT_LOSS_W * boat_type_weights[int(row['boat_type_idx'])]
               + SIDE_LOSS_W * side_weights[int(row['side_idx'])]
               + DIR_LOSS_W  * dir_weights[int(row['dir_idx'])])
            sample_weights.append(w)

        # Stack individual images into a batch array of shape (batch, H, W, 3)
        X = np.stack(images)

        # Labels dict: keys must match the names of the output layers in the model
        y = {
            'boat_output': np.array(boat_labels, dtype=np.int32),
            # side uses one-hot because SparseFocalLoss expects integer targets
            # but side_output here is categorical for the single-head model comparison
            'side_output': keras.utils.to_categorical(side_labels, len(SIDE_CLASSES)),
            'dir_output':  np.array(dir_labels, dtype=np.int32),
        }

        sw = np.array(sample_weights, dtype=np.float32)   # sample weights array
        return X, y, sw

    def on_epoch_end(self):
        # Reshuffle indices after every epoch to prevent the model seeing
        # batches in the same order each time
        if self.shuffle:
            np.random.shuffle(self.indices)


# Instantiate the three generators
train_gen = SailingSequence(DATA_DIR / 'train.csv', augment=True,  shuffle=True)
val_gen   = SailingSequence(DATA_DIR / 'val.csv',   augment=False, shuffle=False)
test_gen  = SailingSequence(DATA_DIR / 'test.csv',  augment=False, shuffle=False)

print('Data generators ready')
print(f'  train={len(train_gen.df):,}  val={len(val_gen.df):,}  test={len(test_gen.df):,} samples')


## 2. Model Definitions

We build and compare **two architectures** that share the same EfficientNetV2B2 backbone.

---

### Model A: Single-Head

The simplest possible approach. After the shared backbone and neck, a single Dense layer outputs all three predictions at the same time using a concatenated softmax. All three tasks (boat type, side, direction) are treated as one big multi-label problem.

```
Input (224x224x3)
      |
 EfficientNetV2B2 backbone
      |
 Neck: Conv2D -> BN -> ReLU -> GlobalMaxPool -> Dense(512) -> BN -> ReLU -> Dropout
      |
 Concatenated output Dense(n_boat + n_side + n_dir) -> split into three softmaxes
```

---

### Model B: Triple-Head (Cascaded)

A more structured approach that models the natural hierarchy of sailing classes. Boat type is easiest to classify, then side (port/starboard), and direction is most fine-grained. Each head sees the shared representation plus an embedding of the previous head's prediction, so the direction head knows what kind of boat it is looking at and which tack it is on.

```
Input (224x224x3)
      |
 EfficientNetV2B2 backbone
      |
 Neck (same as above) -> shared vector
      |
  +---+-------------------+
  |                       |
Head 1: boat_output       |
  |                       |
  boat_embed (64)         |
  |                       |
Head 2: side_output <-- shared + boat_embed
  |
  side_embed (64)
  |
Head 3: dir_output <-- shared + boat_embed + side_embed
```


In [ ]:
# --- Focal Loss --------------------------------------------------------------
# Standard cross-entropy loss treats all misclassified samples equally.
# Focal loss down-weights easy examples so the model concentrates on hard ones.
# gamma=2.0 is a common starting point; higher gamma = more focus on hard samples.
#
# SparseFocalLoss accepts integer class indices (not one-hot vectors).
# Reference: Lin et al. 2017 "Focal Loss for Dense Object Detection"

class SparseFocalLoss(keras.losses.Loss):
    def __init__(self, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING,
                 name='focal_loss', **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma           = gamma          # focusing exponent
        self.label_smoothing = label_smoothing  # softens targets

    def call(self, y_true, y_pred):
        # y_true: integer class indices, shape (batch,)
        # y_pred: softmax probabilities, shape (batch, n_classes)
        n_classes = tf.shape(y_pred)[-1]

        # Convert integer indices to one-hot vectors
        y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), n_classes)

        # Apply label smoothing: spread a small amount of probability to wrong classes
        if self.label_smoothing > 0:
            y_true_oh = (y_true_oh * (1.0 - self.label_smoothing)
                         + self.label_smoothing / tf.cast(n_classes, tf.float32))

        # Clip predictions to avoid log(0)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        # Cross-entropy term: -sum(y_true * log(y_pred))
        ce = -tf.reduce_sum(y_true_oh * tf.math.log(y_pred), axis=-1)

        # Probability of the true class (used to compute the focal weight)
        pt = tf.reduce_sum(y_true_oh * y_pred, axis=-1)

        # Focal weight: (1 - pt)^gamma. Near-zero when the sample is easy (pt close to 1)
        focal_weight = tf.pow(1.0 - pt, self.gamma)

        return focal_weight * ce


print('SparseFocalLoss defined')


In [ ]:
# --- Shared backbone and neck builder ----------------------------------------
# Both models use the same backbone and neck structure so that any difference
# in performance is due to the head architecture, not the feature extractor.

def build_backbone(img_size=IMG_SIZE, trainable=False):
    """
    Load EfficientNetV2B2 pretrained on ImageNet.
    include_top=False means we remove the original classification head.
    trainable=False freezes the backbone weights during the first training phase.
    """
    backbone = EfficientNetV2B2(
        include_top=False,
        weights='imagenet',
        input_shape=(img_size, img_size, 3),
        include_preprocessing=False,   # we apply normalization in the generator
    )
    backbone.trainable = trainable
    return backbone


def build_neck(x, dropout=DROPOUT, weight_decay=WEIGHT_DECAY):
    """
    Neck: takes the backbone feature maps and produces a single shared vector.
    A 1x1 convolution reduces the number of channels before pooling.
    """
    reg = regularizers.l2(weight_decay)                         # L2 weight penalty

    x = layers.Conv2D(256, 1, padding='same',
                      kernel_regularizer=reg, name='neck_conv')(x)  # reduce channels to 256
    x = layers.BatchNormalization(name='neck_bn')(x)            # normalize activations
    x = layers.Activation('relu', name='neck_relu')(x)          # non-linearity
    x = layers.GlobalMaxPooling2D(name='global_pool')(x)        # collapse spatial dims -> vector

    x = layers.Dense(512, kernel_regularizer=reg, name='neck_dense')(x)  # fully connected layer
    x = layers.BatchNormalization(name='neck_dense_bn')(x)
    x = layers.Activation('relu', name='neck_dense_relu')(x)
    x = layers.Dropout(dropout, name='neck_dropout')(x)         # dropout for regularization

    return x   # shape: (batch, 512)


print('Backbone and neck builders defined')


In [ ]:
# =============================================================================
# MODEL A: Single-Head
# =============================================================================
# This model concatenates all three outputs into one prediction.
# It is the simplest baseline: one Dense layer produces logits for all classes
# simultaneously, then three separate softmaxes are applied.
#
# The downside is that it cannot model the natural ordering of tasks:
# boat type and side are simpler and should inform direction, but this
# architecture does not allow that.

def build_single_head_model(n_boat, n_side, n_dir,
                             img_size=IMG_SIZE, dropout=DROPOUT,
                             weight_decay=WEIGHT_DECAY, trainable_backbone=False):

    reg      = regularizers.l2(weight_decay)
    backbone = build_backbone(img_size, trainable=trainable_backbone)

    # Define the model's input tensor
    inputs = keras.Input(shape=(img_size, img_size, 3), name='image_input')

    # Rescale from [0,255] to [0,1] inside the model (defensive, since we normalize in generator too)
    x = layers.Rescaling(1.0 / 255.0, name='rescaling')(inputs)

    # Pass through the pretrained backbone to extract feature maps
    x = backbone(x)

    # Build the shared neck
    shared = build_neck(x, dropout=dropout, weight_decay=weight_decay)

    # Single combined prediction head
    # Produces logits for all three tasks at the same time
    combined = layers.Dense(256, kernel_regularizer=reg,
                            activation='relu', name='shared_dense_a')(shared)  # intermediate layer
    combined = layers.Dropout(dropout * 0.5, name='shared_dropout_a')(combined)

    # Three separate softmax outputs, each treating its task independently
    boat_out = layers.Dense(n_boat, activation='softmax',
                            name='boat_output')(combined)   # boat type prediction
    side_out = layers.Dense(n_side, activation='softmax',
                            name='side_output')(combined)   # side (port/starboard) prediction
    dir_out  = layers.Dense(n_dir,  activation='softmax',
                            name='dir_output')(combined)    # sailing direction prediction

    return keras.Model(
        inputs=inputs,
        outputs=[boat_out, side_out, dir_out],
        name='model_A_single_head'
    ), backbone


# Build Model A
model_A, backbone_A = build_single_head_model(
    n_boat=len(BOAT_TYPE_CLASSES),
    n_side=len(SIDE_CLASSES),
    n_dir=len(DIRECTION_CLASSES),
)

print('Model A (single-head) built')
print(f'  Total parameters: {model_A.count_params()/1e6:.1f}M')
model_A.summary()


In [ ]:
# =============================================================================
# MODEL B: Triple-Head (Cascaded)
# =============================================================================
# This model uses three separate heads arranged in a cascade.
# Each head passes a small embedding to the next one so downstream tasks
# can condition on the predictions of upstream tasks.
#
# Order: boat_type -> side -> direction
# Rationale: direction depends on both the boat type and the tack (side),
# so the direction head benefits from knowing both.
#
# stop_gradient is applied to the embedding inputs so the gradient from
# head N does not flow back through head N-1.
# This prevents later heads from destabilizing earlier heads during training.

def build_triple_head_model(n_boat, n_side, n_dir,
                             img_size=IMG_SIZE, dropout=DROPOUT,
                             weight_decay=WEIGHT_DECAY, trainable_backbone=False):

    reg      = regularizers.l2(weight_decay)
    backbone = build_backbone(img_size, trainable=trainable_backbone)

    inputs = keras.Input(shape=(img_size, img_size, 3), name='image_input')

    x = layers.Rescaling(1.0 / 255.0, name='rescaling')(inputs)

    x = backbone(x)

    shared = build_neck(x, dropout=dropout, weight_decay=weight_decay)

    # --- Head 1: boat_type ---------------------------------------------------
    # The simplest classification: two boat types (ILCA, Optimist)
    boat_out = layers.Dense(n_boat, activation='softmax',
                            name='boat_output')(shared)     # direct prediction from shared vector

    # boat_embed: a 64-dimensional vector derived from the boat_type prediction.
    # stop_gradient blocks gradients so Head 2 cannot modify Head 1's weights.
    boat_embed = layers.Dense(64, activation='relu', name='boat_embed')(
        keras.ops.stop_gradient(boat_out))                  # (batch, 64)

    # --- Head 2: side --------------------------------------------------------
    # Concatenate the shared vector with the boat embedding
    side_neck = layers.Concatenate(name='side_neck_concat')([shared, boat_embed])
    side_neck = layers.Dense(256, kernel_regularizer=reg,
                             name='side_dense')(side_neck)  # learn from both features
    side_neck = layers.BatchNormalization(name='side_bn')(side_neck)
    side_neck = layers.Activation('relu', name='side_relu')(side_neck)
    side_neck = layers.Dropout(dropout * 0.5, name='side_dropout')(side_neck)
    side_out  = layers.Dense(n_side, activation='softmax',
                             name='side_output')(side_neck)

    # side_embed: same idea as boat_embed, passes side info to the direction head
    side_embed = layers.Dense(64, activation='relu', name='side_embed')(
        keras.ops.stop_gradient(side_out))                  # (batch, 64)

    # --- Head 3: direction ---------------------------------------------------
    # The hardest task: 6 sailing directions.
    # Sees the shared features plus both boat and side embeddings.
    dir_neck = layers.Concatenate(name='dir_neck_concat')([shared, boat_embed, side_embed])
    dir_neck = layers.Dense(256, kernel_regularizer=reg,
                            name='dir_dense')(dir_neck)
    dir_neck = layers.BatchNormalization(name='dir_bn')(dir_neck)
    dir_neck = layers.Activation('relu', name='dir_relu')(dir_neck)
    dir_neck = layers.Dropout(dropout * 0.5, name='dir_dropout')(dir_neck)
    dir_out  = layers.Dense(n_dir, activation='softmax',
                            name='dir_output')(dir_neck)

    return keras.Model(
        inputs=inputs,
        outputs=[boat_out, side_out, dir_out],
        name='model_B_triple_head'
    ), backbone


# Build Model B
model_B, backbone_B = build_triple_head_model(
    n_boat=len(BOAT_TYPE_CLASSES),
    n_side=len(SIDE_CLASSES),
    n_dir=len(DIRECTION_CLASSES),
)

print('Model B (triple-head) built')
print(f'  Total parameters: {model_B.count_params()/1e6:.1f}M')
model_B.summary()


In [ ]:
# Save architecture diagrams for inspection
keras.utils.plot_model(model_A, to_file=str(RESULTS_DIR / 'model_A.png'), show_shapes=True)
keras.utils.plot_model(model_B, to_file=str(RESULTS_DIR / 'model_B.png'), show_shapes=True)
print('Architecture diagrams saved')


## 3. Training

Both models are trained with the same procedure so the comparison is fair:

1. **Phase 1 — frozen backbone** (first `FREEZE_EPOCHS` epochs): only the head weights are trained.
   This lets the heads stabilize before the backbone starts to adapt.
2. **Phase 2 — fine-tuning** (remaining epochs): the backbone is unfrozen and all weights are trained
   with a lower learning rate `LR_FINE`.

We use:
- `SparseFocalLoss` for boat and direction (integer targets), standard categorical cross-entropy for side (one-hot target)
- `EarlyStopping` with patience=`EARLY_STOP_PAT`: stops training and restores the best weights
- `ReduceLROnPlateau`: halves the learning rate when validation loss stops improving


In [ ]:
def compile_model(model, lr):
    """
    Compile a model with the chosen losses, optimizer, and metrics.
    This function is called twice per model: once for Phase 1 and once for Phase 2,
    because we change the learning rate between phases.
    """
    # Adam with weight decay is a common choice for computer vision
    optimizer = keras.optimizers.Adam(learning_rate=lr, weight_decay=WEIGHT_DECAY)

    model.compile(
        optimizer=optimizer,

        # Loss function for each output head (keys must match output layer names)
        loss={
            'boat_output': SparseFocalLoss(gamma=FOCAL_GAMMA),     # integer targets
            'side_output': 'categorical_crossentropy',             # one-hot targets
            'dir_output':  SparseFocalLoss(gamma=FOCAL_GAMMA),     # integer targets
        },

        # Loss weights: direction is the hardest task so it has the highest weight
        loss_weights={
            'boat_output': BOAT_LOSS_W,
            'side_output': SIDE_LOSS_W,
            'dir_output':  DIR_LOSS_W,
        },

        # Track accuracy for each head separately
        metrics={
            'boat_output': ['accuracy'],
            'side_output': ['accuracy'],
            'dir_output':  ['accuracy'],
        },
    )


def get_callbacks(checkpoint_path):
    """
    Return the list of Keras callbacks used during training.
    checkpoint_path: where to save the best model weights.
    """
    return [
        # Save the model whenever the validation loss improves
        keras.callbacks.ModelCheckpoint(
            filepath=str(checkpoint_path),
            monitor='val_loss',
            save_best_only=True,       # only overwrite if this is the best so far
            verbose=1,
        ),

        # Stop training early if val_loss does not improve for EARLY_STOP_PAT epochs
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=EARLY_STOP_PAT,
            restore_best_weights=True, # revert to the best weights when stopping
            verbose=1,
        ),

        # Halve the learning rate when val_loss plateaus for 3 epochs
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-6,
            verbose=1,
        ),

        # Log all epoch metrics to a CSV file for later inspection
        keras.callbacks.CSVLogger(str(RESULTS_DIR / checkpoint_path.stem + '_log.csv')),
    ]


print('Compile and callback helpers defined')


In [ ]:
def train_model(model, backbone, model_name):
    """
    Full two-phase training procedure for a given model.
    Returns the combined training history.
    """
    checkpoint_path = CHECKPOINT_DIR / f'{model_name}.keras'

    print(f'\n--- Training {model_name} ---')

    # ---- Phase 1: train heads only, backbone frozen -------------------------
    print(f'Phase 1: backbone frozen, {FREEZE_EPOCHS} epochs, lr={LR_INITIAL}')
    backbone.trainable = False    # freeze all backbone layers
    compile_model(model, lr=LR_INITIAL)

    history1 = model.fit(
        train_gen,                # training data generator
        validation_data=val_gen,  # validation data generator
        epochs=FREEZE_EPOCHS,     # only run for FREEZE_EPOCHS epochs in this phase
        callbacks=get_callbacks(checkpoint_path),
        verbose=1,
    )

    # ---- Phase 2: fine-tune with a lower learning rate ----------------------
    print(f'Phase 2: backbone unfrozen, up to {NUM_EPOCHS} epochs, lr={LR_FINE}')
    backbone.trainable = True     # unfreeze backbone weights
    compile_model(model, lr=LR_FINE)

    history2 = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=NUM_EPOCHS,        # EarlyStopping will stop this before NUM_EPOCHS
        callbacks=get_callbacks(checkpoint_path),
        verbose=1,
    )

    # Merge the two history dicts so we can plot a single training curve
    combined_history = {}
    for key in history1.history:
        combined_history[key] = history1.history[key] + history2.history[key]

    return combined_history


# Train both models
history_A = train_model(model_A, backbone_A, 'model_A_single_head')
history_B = train_model(model_B, backbone_B, 'model_B_triple_head')

print('\nTraining complete')


In [ ]:
# --- Plot training curves for both models ------------------------------------
# We plot the total validation loss over epochs to see how quickly each model
# converges and whether one overfits more than the other.

def plot_training_curves(history_A, history_B):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Training Curves: Model A vs Model B', fontweight='bold')

    # Left: total loss
    ax = axes[0]
    ax.plot(history_A['val_loss'],  label='Model A val loss',  color='steelblue',  lw=2)
    ax.plot(history_A['loss'],      label='Model A train loss', color='steelblue',  lw=1, ls='--')
    ax.plot(history_B['val_loss'],  label='Model B val loss',  color='darkorange', lw=2)
    ax.plot(history_B['loss'],      label='Model B train loss', color='darkorange', lw=1, ls='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Total Loss')
    ax.legend()

    # Right: direction accuracy (the hardest and most important task)
    ax = axes[1]
    key = 'val_dir_output_accuracy'
    if key in history_A and key in history_B:
        ax.plot(history_A[key], label='Model A dir accuracy',  color='steelblue',  lw=2)
        ax.plot(history_B[key], label='Model B dir accuracy',  color='darkorange', lw=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.set_title('Direction Accuracy (Validation)')
        ax.legend()

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'training_curves_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Training curves saved')


plot_training_curves(history_A, history_B)


## 4. Evaluation and Comparison

We evaluate both models on the **test set** (unseen during training) and compare:
- Accuracy per head (boat type, side, direction)
- Confusion matrices for the direction head (most informative)


In [ ]:
def evaluate_model(model, test_gen, model_name):
    """
    Run the model on the full test set.
    Returns a dict with accuracy for each head and the raw predictions.
    """
    print(f'\nEvaluating {model_name} on test set...')

    # Collect ground-truth labels and predictions over all test batches
    true_boat, true_side, true_dir   = [], [], []
    pred_boat, pred_side, pred_dir   = [], [], []

    for batch_idx in range(len(test_gen)):
        X, y, _        = test_gen[batch_idx]             # get one test batch

        # Run forward pass; model returns three arrays, one per output head
        p_boat, p_side, p_dir = model.predict(X, verbose=0)

        # Convert softmax probabilities to integer class indices (argmax)
        pred_boat.extend(np.argmax(p_boat, axis=1))
        pred_side.extend(np.argmax(p_side, axis=1))
        pred_dir.extend( np.argmax(p_dir,  axis=1))

        # Ground truth: boat and dir are already integers; side is one-hot so we argmax too
        true_boat.extend(y['boat_output'].tolist())
        true_side.extend(np.argmax(y['side_output'], axis=1).tolist())
        true_dir.extend( y['dir_output'].tolist())

    # Convert lists to numpy arrays
    true_boat = np.array(true_boat); pred_boat = np.array(pred_boat)
    true_side = np.array(true_side); pred_side = np.array(pred_side)
    true_dir  = np.array(true_dir);  pred_dir  = np.array(pred_dir)

    # Compute accuracy for each head
    acc_boat = (true_boat == pred_boat).mean()
    acc_side = (true_side == pred_side).mean()
    acc_dir  = (true_dir  == pred_dir ).mean()

    print(f'  {model_name} | boat acc={acc_boat:.3f} | side acc={acc_side:.3f} | dir acc={acc_dir:.3f}')

    return {
        'acc_boat': acc_boat, 'acc_side': acc_side, 'acc_dir': acc_dir,
        'true_boat': true_boat, 'pred_boat': pred_boat,
        'true_side': true_side, 'pred_side': pred_side,
        'true_dir':  true_dir,  'pred_dir':  pred_dir,
    }


results_A = evaluate_model(model_A, test_gen, 'Model A (single-head)')
results_B = evaluate_model(model_B, test_gen, 'Model B (triple-head)')


In [ ]:
# --- Side-by-side accuracy table -------------------------------------------
# A simple table to summarize which model is better on each task

print('\n=== Accuracy Comparison ===')
print(f'{"Task":<15}  {"Model A":>10}  {"Model B":>10}  {"Winner":>10}')
print('-' * 50)

for task, key in [('Boat type', 'acc_boat'), ('Side', 'acc_side'), ('Direction', 'acc_dir')]:
    a = results_A[key]    # accuracy of Model A on this task
    b = results_B[key]    # accuracy of Model B on this task
    winner = 'Model A' if a > b else ('Model B' if b > a else 'Tie')
    print(f'{task:<15}  {a:>10.3f}  {b:>10.3f}  {winner:>10}')


In [ ]:
# --- Confusion matrices for the direction head --------------------------------
# The direction head is the most important and most difficult, so we inspect
# its confusion matrix to see which classes are commonly confused.

def plot_confusion_matrices(results_A, results_B, classes, task_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'Confusion Matrix — {task_name}', fontweight='bold')

    for ax, results, title in [
        (axes[0], results_A, 'Model A (single-head)'),
        (axes[1], results_B, 'Model B (triple-head)'),
    ]:
        # Compute the confusion matrix: rows=true labels, cols=predicted labels
        cm = confusion_matrix(results[f'true_{task_name.lower()[:3]}'],
                              results[f'pred_{task_name.lower()[:3]}'])

        # Normalize to get fractions instead of raw counts (easier to compare across class sizes)
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

        im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap='Blues')
        ax.set_title(title)
        ax.set_xticks(range(len(classes)))
        ax.set_yticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=40, ha='right', fontsize=9)
        ax.set_yticklabels(classes, fontsize=9)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

        # Print the value inside each cell of the matrix
        for i in range(len(classes)):
            for j in range(len(classes)):
                ax.text(j, i, f'{cm_norm[i,j]:.2f}',
                        ha='center', va='center',
                        color='white' if cm_norm[i,j] > 0.5 else 'black',
                        fontsize=8)

        plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'confusion_{task_name.lower()}.png', dpi=150, bbox_inches='tight')
    plt.show()


# Plot for all three tasks
plot_confusion_matrices(results_A, results_B,
    {k: v for k, v in IDX2BOAT.items()}.values() if False else list(BOAT_TYPE_CLASSES),
    'Boat')   # boat_type confusion matrix

plot_confusion_matrices(results_A, results_B,
    SIDE_CLASSES, 'Side')    # side confusion matrix

plot_confusion_matrices(results_A, results_B,
    DIRECTION_CLASSES, 'Dir')    # direction confusion matrix — the most important one


In [ ]:
# --- Print full classification report for both models -----------------------
# sklearn's classification_report shows precision, recall, and F1-score per class.
# This is more informative than accuracy alone when classes are imbalanced.

for label, results in [('Model A (single-head)', results_A),
                        ('Model B (triple-head)', results_B)]:
    print(f'\n=== {label} — Direction Classification Report ===')
    print(classification_report(
        results['true_dir'],
        results['pred_dir'],
        target_names=DIRECTION_CLASSES,
        zero_division=0,    # avoid division by zero if a class has no predictions
    ))


In [ ]:
# --- Save evaluation summary to JSON ----------------------------------------
# This allows comparing results later without re-running the notebook

summary = {
    'model_A': {k: float(v) for k, v in results_A.items() if k.startswith('acc')},
    'model_B': {k: float(v) for k, v in results_B.items() if k.startswith('acc')},
}

with open(RESULTS_DIR / 'eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('Evaluation summary saved to eval_summary.json')
print(json.dumps(summary, indent=2))


## Done

| Output file | Description |
|---|---|
| `checkpoints/model_A_single_head.keras` | Best weights for Model A |
| `checkpoints/model_B_triple_head.keras` | Best weights for Model B |
| `results/model_A.png` | Architecture diagram Model A |
| `results/model_B.png` | Architecture diagram Model B |
| `results/training_curves_comparison.png` | Loss and accuracy curves for both models |
| `results/confusion_Boat.png` | Confusion matrix for boat type |
| `results/confusion_Side.png` | Confusion matrix for side |
| `results/confusion_Dir.png` | Confusion matrix for direction |
| `results/eval_summary.json` | Accuracy numbers for both models |
